In [42]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [15]:
raw=pd.read_csv(r"C:\Users\shrusti\Downloads\task1_marketplace_delivery_sla_raw.csv")

In [16]:
raw

,order_id,customer_id,order_date,shipped_at,delivered_at,updated_at,city,courier,payment_status,order_status,promised_days,order_value,is_test_order
0,O1001,C01,29-04-26,01-05-26,03-05-26,03-05-26,Bengaluru,ShipQuick,Paid,Delivered,2,1200,No
1,O1002,C02,02-05-26,03-05-26,07-05-26,07-05-26,Mumbai,shipquick,pending,delivered,3,900,No
2,O1002,C02,02-05-26,03-05-26,07-05-26,08-05-26,Mumbai,ShipQuick,Paid,Delivered,3,900,No
3,O1003,C03,04-05-26,05-05-26,06-05-26,06-05-26,Delhi,FastBee,Pending,Delivered,2,650,No
4,O1004,C04,05-05-26,06-05-26,01-06-26,01-06-26,Pune,Blue Dart,Paid,Delivered,4,2000,No
5,O1005,C05,08-05-26,09-05-26,NaN,10-05-26,Chennai,Fastbee,Paid,Cancelled,3,700,No
6,O1006,C06,10-05-26,11-05-26,14-05-26,14-05-26,Mumbai,Bluedart,Paid,Delivered,3,1500,No
7,O1006,C06,10-05-26,11-05-26,14-05-26,14-05-26,Mumbai,Bluedart,Paid,Delivered,3,1500,No
8,O1007,C07,12-05-26,13-05-26,15-05-26,15-05-26,Bangalore,ShipQuick,Paid,Delivered,2,1,Yes
9,O1008,C08,15-05-26,16-05-26,18-05-26,18-05-26,Pune,FastBee,Paid,Delivered,-2,500,No


In [17]:
print("RAW SHAPE:", raw.shape)
print(raw)
print()
 
df = raw.copy()

RAW SHAPE: (10, 13)
  order_id customer_id order_date shipped_at delivered_at updated_at  \
0    O1001         C01   29-04-26   01-05-26     03-05-26   03-05-26   
1    O1002         C02   02-05-26   03-05-26     07-05-26   07-05-26   
2    O1002         C02   02-05-26   03-05-26     07-05-26   08-05-26   
3    O1003         C03   04-05-26   05-05-26     06-05-26   06-05-26   
4    O1004         C04   05-05-26   06-05-26     01-06-26   01-06-26   
5    O1005         C05   08-05-26   09-05-26          NaN   10-05-26   
6    O1006         C06   10-05-26   11-05-26     14-05-26   14-05-26   
7    O1006         C06   10-05-26   11-05-26     14-05-26   14-05-26   
8    O1007         C07   12-05-26   13-05-26     15-05-26   15-05-26   
9    O1008         C08   15-05-26   16-05-26     18-05-26   18-05-26   

        city    courier payment_status order_status  promised_days  \
0  Bengaluru  ShipQuick           Paid    Delivered              2   
1     Mumbai  shipquick        pending    deliv

In [18]:
str_cols = df.select_dtypes(include="object").columns
for c in str_cols:
    df[c] = df[c].astype(str).str.strip()
 
date_cols = ["order_date", "shipped_at", "delivered_at", "updated_at"]
for c in date_cols:
    df[c] = pd.to_datetime(df[c], format="%d-%m-%y", errors="coerce")

In [19]:
issues = []
exact_dupes = df.duplicated(subset=[c for c in df.columns if c != "order_id"] + ["order_id"], keep=False)
if exact_dupes.any():
    issues.append(f"{exact_dupes.sum()} fully-duplicated row(s) found (e.g. order_id O1006) -> dropped extra copies.")

In [21]:
dupe_ids = df["order_id"][df["order_id"].duplicated(keep=False)].unique()
if len(dupe_ids):
    issues.append(f"order_id(s) with multiple conflicting records: {list(dupe_ids)} -> kept the row with the latest updated_at as the source of truth.")

In [22]:
for c in ["courier", "payment_status", "order_status", "is_test_order"]:
    variants = raw[c].astype(str).str.strip().unique()
    normalized_variants = pd.Series(variants).str.lower().str.replace(" ", "", regex=False).unique()
    if len(variants) != len(normalized_variants):
        issues.append(f"Inconsistent text casing/spacing in '{c}': {sorted(variants)} -> normalized to a single canonical label per value.")

In [23]:
missing_delivered = df["delivered_at"].isna().sum()
if missing_delivered:
    issues.append(f"{missing_delivered} order(s) missing delivered_at (e.g. cancelled order O1005) -> excluded, can't compute transit time.")

In [24]:
neg_promised = (df["promised_days"] < 0).sum()
if neg_promised:
    issues.append(f"{neg_promised} order(s) with a negative promised_days value (e.g. O1008 = -2) -> treated as invalid/unreliable SLA data and excluded.")

In [25]:
test_orders = (df["is_test_order"].str.lower() == "yes").sum()
if test_orders:
    issues.append(f"{test_orders} order(s) flagged is_test_order = Yes -> excluded per column definition (not real deliveries).")

In [26]:
non_may = df["delivered_at"].notna() & ~((df["delivered_at"].dt.year == 2026) & (df["delivered_at"].dt.month == 5))
if non_may.any():
    issues.append(f"{non_may.sum()} order(s) delivered outside May 2026 (e.g. O1004 delivered 01-06-26) -> excluded from this May-2026 analysis.")
 
print("DATA QUALITY ISSUES FOUND:")
for i, msg in enumerate(issues, 1):
    print(f"  {i}. {msg}")
print()

DATA QUALITY ISSUES FOUND:
  1. 2 fully-duplicated row(s) found (e.g. order_id O1006) -> dropped extra copies.
  2. order_id(s) with multiple conflicting records: ['O1002', 'O1006'] -> kept the row with the latest updated_at as the source of truth.
  3. Inconsistent text casing/spacing in 'courier': ['Blue Dart', 'Bluedart', 'FastBee', 'Fastbee', 'ShipQuick', 'shipquick'] -> normalized to a single canonical label per value.
  4. Inconsistent text casing/spacing in 'payment_status': ['Paid', 'Pending', 'pending'] -> normalized to a single canonical label per value.
  5. Inconsistent text casing/spacing in 'order_status': ['Cancelled', 'Delivered', 'delivered'] -> normalized to a single canonical label per value.
  6. 1 order(s) missing delivered_at (e.g. cancelled order O1005) -> excluded, can't compute transit time.
  7. 1 order(s) with a negative promised_days value (e.g. O1008 = -2) -> treated as invalid/unreliable SLA data and excluded.
  8. 1 order(s) flagged is_test_order = Yes ->

In [29]:
def canon(series, mapping=None):
    s = series.str.strip()
    key = s.str.lower().str.replace(" ", "", regex=False)
    if mapping:
        return key.map(mapping).fillna(s)
    return key
 
courier_map = {
    "shipquick": "ShipQuick",
    "fastbee": "FastBee",
    "bluedart": "Blue Dart",
}
df["courier_clean"] = canon(df["courier"], courier_map)
df["payment_status_clean"] = df["payment_status"].str.strip().str.capitalize()
df["order_status_clean"] = df["order_status"].str.strip().str.capitalize()
df["is_test_order_clean"] = df["is_test_order"].str.strip().str.capitalize()

In [31]:
df = df.sort_values("updated_at").drop_duplicates(subset="order_id", keep="last").reset_index(drop=True)
 

In [32]:
df["actual_transit_days"] = (df["delivered_at"] - df["shipped_at"]).dt.days
 
eligible = df[
    (df["order_status_clean"] == "Delivered")
    & (df["is_test_order_clean"] == "No")
    & (df["delivered_at"].notna())
    & (df["delivered_at"].dt.year == 2026)
    & (df["delivered_at"].dt.month == 5)
    & (df["promised_days"] > 0)
].copy()
 
eligible["delivery_status"] = np.where(
    eligible["actual_transit_days"] <= eligible["promised_days"], "On Time", "Late"
)
 
print("ELIGIBLE ORDER-LEVEL DETAIL:")
print(eligible[["order_id", "courier_clean", "shipped_at", "delivered_at",
                 "actual_transit_days", "promised_days", "delivery_status"]])
print()

ELIGIBLE ORDER-LEVEL DETAIL:
  order_id courier_clean shipped_at delivered_at  actual_transit_days  \
0    O1001     ShipQuick 2026-05-01   2026-05-03                  2.0   
1    O1003       FastBee 2026-05-05   2026-05-06                  1.0   
2    O1002     ShipQuick 2026-05-03   2026-05-07                  4.0   
4    O1006     Blue Dart 2026-05-11   2026-05-14                  3.0   

   promised_days delivery_status  
0              2         On Time  
1              2         On Time  
2              3            Late  
4              3         On Time  



In [38]:
summary = (
    eligible.groupby("courier_clean")
    .agg(
        eligible_orders=("order_id", "count"),
        on_time_orders=("delivery_status", lambda s: (s == "On Time").sum()),
    )
    .reset_index()
    .rename(columns={"courier_clean": "courier"})
)
summary["on_time_rate"] = (summary["on_time_orders"] / summary["eligible_orders"]).round(2)
summary = summary.sort_values("on_time_rate", ascending=False).reset_index(drop=True)
 
print("COURIER-LEVEL ON-TIME DELIVERY SUMMARY (May 2026):")
print(summary.to_string(index=False))
 
summary.to_csv("C:\\Users\\shrusti\\Downloads\\courier_summary.csv", index=False)
eligible.to_csv("C:\\Users\\shrusti\\Downloads\\eligible_orders_detail.csv", index=False)

COURIER-LEVEL ON-TIME DELIVERY SUMMARY (May 2026):
  courier  eligible_orders  on_time_orders  on_time_rate
Blue Dart                1               1           1.0
  FastBee                1               1           1.0
ShipQuick                2               1           0.5


In [43]:
plt.style.use("default")

In [46]:
fig1, ax1 = plt.subplots(figsize=(7, 4.5))
bars = ax1.bar(summary["courier"], summary["on_time_rate"], color="#4C72B0")
ax1.set_ylim(0, 1.1)
ax1.set_ylabel("On-Time Rate")
ax1.set_title("On-Time Delivery Rate by Courier — May 2026")
for b, rate, n in zip(bars, summary["on_time_rate"], summary["eligible_orders"]):
    ax1.text(b.get_x() + b.get_width()/2, b.get_height() + 0.03, f"{rate:.0%}\n(n={n})",
              ha="center", fontsize=9)
ax1.axhline(1.0, color="grey", linestyle="--", linewidth=0.8)
plt.tight_layout()
fig1.savefig("C:\\Users\\shrusti\\Downloads\\chart1_ontime_rate_by_courier.png", dpi=150)
plt.close(fig1)
 


In [48]:
counts = eligible.groupby(["courier_clean", "delivery_status"]).size().unstack(fill_value=0)
fig2, ax2 = plt.subplots(figsize=(7, 4.5))
counts.plot(kind="bar", stacked=True, ax=ax2, color={"On Time": "#55A868", "Late": "#C44E52"})
ax2.set_title("Eligible Orders: On Time vs Late by Courier — May 2026")
ax2.set_xlabel("Courier")
ax2.set_ylabel("Number of Orders")
plt.xticks(rotation=0)
plt.tight_layout()
fig2.savefig("C:\\Users\\shrusti\\Downloads\\chart2_ontime_vs_late_counts.png", dpi=150)
plt.close(fig2)

In [50]:
exclusion_reasons = {
    "Duplicate order_id\n(kept latest record)": (raw["order_id"].duplicated(keep="first")).sum(),
    "Missing delivered_at\n(e.g. cancelled)": df["delivered_at"].isna().sum(),
    "Test order": (df["is_test_order_clean"] == "Yes").sum(),
    "Invalid promised_days\n(negative)": (df["promised_days"] <= 0).sum(),
    "Delivered outside\nMay 2026": non_may.sum(),
}
fig3, ax3 = plt.subplots(figsize=(7, 4.5))
ax3.barh(list(exclusion_reasons.keys()), list(exclusion_reasons.values()), color="#DD8452")
ax3.set_title("Orders Excluded from May-2026 On-Time Analysis, by Reason")
ax3.set_xlabel("Number of Orders")
plt.tight_layout()
fig3.savefig("C:\\Users\\shrusti\\Downloads\\chart3_exclusion_reasons.png", dpi=150)
plt.close(fig3)
 
print("\nCharts saved.")


Charts saved.
